<a href="https://colab.research.google.com/github/Debapri/git-advanced/blob/main/Data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Robust Colab loader + quick cleaner with diagnostics
import pandas as pd, numpy as np, os, traceback, csv, zipfile
from google.colab import files

print("📁 Upload your dataset now (CSV / XLSX / ZIP).")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print("Uploaded file:", filename)

def show_file_head(fname, n=5):
    print("\n--- Raw first lines (bytes) ---")
    try:
        with open(fname, 'rb') as f:
            for i, line in enumerate(f.readlines()[:n]):
                print(i+1, line[:500])
    except Exception as e:
        print("Couldn't read raw lines:", e)

def try_read(fname):
    attempts = []
    lower = fname.lower()

    # Excel
    if lower.endswith(('.xlsx', '.xls')):
        try:
            df = pd.read_excel(fname)
            return df, "pd.read_excel"
        except Exception:
            attempts.append(("pd.read_excel", traceback.format_exc()))

    # ZIP (try to find a CSV inside)
    if lower.endswith('.zip'):
        try:
            with zipfile.ZipFile(fname) as z:
                files_in_zip = z.namelist()
                print("ZIP contains:", files_in_zip)
                for f in files_in_zip:
                    if f.lower().endswith('.csv'):
                        z.extract(f)
                        df = pd.read_csv(f, on_bad_lines='skip', engine='python')
                        return df, f"zip->{f}"
            attempts.append(("zip", "no csv file found inside"))
        except Exception:
            attempts.append(("zip", traceback.format_exc()))

    # Try default CSV read
    try:
        df = pd.read_csv(fname)
        return df, "pd.read_csv (default)"
    except Exception:
        attempts.append(("pd.read_csv default", traceback.format_exc()))

    # Try multiple encodings
    for enc in ('utf-8','latin1','iso-8859-1','cp1252'):
        try:
            df = pd.read_csv(fname, encoding=enc)
            return df, f"pd.read_csv encoding={enc}"
        except Exception:
            attempts.append((f"pd.read_csv encoding={enc}", traceback.format_exc()))

    # Try delimiter sniffing
    try:
        with open(fname, 'r', encoding='utf-8', errors='replace') as f:
            sample = f.read(2048)
        dialect = csv.Sniffer().sniff(sample, delimiters=[',',';','\t','|'])
        sep = dialect.delimiter
        try:
            df = pd.read_csv(fname, sep=sep, engine='python', on_bad_lines='skip')
            return df, f"pd.read_csv sep={sep}"
        except Exception:
            attempts.append((f"sniffer sep={sep}", traceback.format_exc()))
    except Exception:
        attempts.append(("sniffer", "could not sniff delimiter"))

    # Final fallback
    try:
        df = pd.read_table(fname, engine='python', on_bad_lines='skip')
        return df, "pd.read_table fallback"
    except Exception:
        attempts.append(("pd.read_table", traceback.format_exc()))

    return None, attempts

# Show raw preview
show_file_head(filename)

# Attempt reads
df, info = try_read(filename)
if df is None:
    print("\n❌ Could not read the file. Attempts and tracebacks:")
    for tag, tb in info:
        print("\n--- Attempt:", tag, "---")
        print(tb)
    raise SystemExit("Reading failed. Inspect tracebacks above and adjust file/format.")

print("\n✅ File loaded using:", info)
print("Shape:", df.shape)
print(df.head())

# ===== Quick, safe cleaning =====
df = df.drop_duplicates()

# Convert numeric-ish columns safely
for col in df.select_dtypes(include=['object','int','float']).columns:
    # try to coerce columns that contain numeric-like strings
    try:
        coerced = pd.to_numeric(df[col].astype(str).str.replace(r'[^\d\.-]', '', regex=True), errors='coerce')
        # if many values became numeric (heuristic), keep numeric
        if coerced.notna().sum() > (len(df) * 0.3):
            df[col] = coerced
    except Exception:
        pass

# Numeric columns: fill median
num_cols = df.select_dtypes(include='number').columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Text columns: safe string ops
obj_cols = df.select_dtypes(include='object').columns
for col in obj_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col].replace({'nan':'unknown','none':'unknown'}, inplace=True)
    df[col].fillna('unknown', inplace=True)

# Date-like columns: try to parse
for col in df.columns:
    if 'date' in col.lower() or 'time' in col.lower():
        df[col] = pd.to_datetime(df[col], errors='coerce')

print("\n--- After cleaning ---")
print(df.info())
print(df.head())

# Save and provide download
out_name = 'cleaned_data.csv'
df.to_csv(out_name, index=False)
print(f"\n✅ Cleaned file saved to {out_name}. Downloading...")
files.download(out_name)


📁 Upload your dataset now (CSV / XLSX / ZIP).


Saving customers-100.csv.xlsx to customers-100.csv (3).xlsx
Uploaded file: customers-100.csv (3).xlsx

--- Raw first lines (bytes) ---
1 b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x00b\xee\x9dh^\x01\x00\x00\x90\x04\x00\x00\x13\x00\x08\x02[Content_Types].xml \xa2\x04\x02(\xa0\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

/tmp/ipython-input-2988450300.py:117: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipython-input-2988450300.py:123: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>